In [ ]:
import pandas as pd

# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\modified_csv_May.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')


# Display the DataFrame
print("Contents of the CSV file:")
print(df)

In [ ]:
# Extract the first 4 rows
df_first_four_rows = df.head(4)

# Path to save the new CSV file
new_csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\new_small_csv_June.csv'

# Save the DataFrame with the first 4 rows to a new CSV file
df_first_four_rows.to_csv(new_csv_path, index=False)

# Display the DataFrame
print("Contents of the CSV file with the first 4 rows:")
print(df_first_four_rows)

In [ ]:
import os
import time
import datetime
import csv
import json
import elabapi_python
from elabapi_python.rest import ApiException
import pandas as pd

#########################
#         CONFIG        #
#########################
API_HOST_URL = 'https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2'
# replace with your api key
API_KEY = '34-29d13c485c893e66cfda134c1910df50c7bbdf1915d5f8cad7d7912f5cd1c017765f3f9ff3bd733a70d334'
#########################
#      END CONFIG       #
#########################

# Configure the api client
configuration = elabapi_python.Configuration()
configuration.api_key['api_key'] = API_KEY
configuration.api_key_prefix['api_key'] = 'Authorization'
configuration.host = API_HOST_URL
configuration.debug = False
configuration.verify_ssl = False

# Create an instance of the API class
api_client = elabapi_python.ApiClient(configuration)
# Fix issue with Authorization header not being properly set by the generated lib
api_client.set_default_header(header_name='Authorization', header_value=API_KEY)

# Load the items API
items_api = elabapi_python.ItemsApi(api_client)


In [ ]:
# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\new_small_csv_June.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')

# Category ID in eLab where the items will be created
category_id = 238  # Adjust this according to your category ID in eLab

# Base JSON for metadata
base_json = '''{
    "elabftw": {
        "display_main_text": false
    },
    "extra_fields": {
        "Inventory number": {
            "type": "text",
            "value": ""
        },
        "Substance name": {
            "type": "text",
            "value": ""
        },
        "Location": {
            "type": "text",
            "value": ""
        },
        "Purity": {
            "type": "text",
            "value": ""
        },
        "Manufacturer": {
            "type": "text",
            "value": ""
        },
        "CAS number": {
            "type": "text",
            "value": ""
        },
        "Batch number": {
            "type": "text",
            "value": ""
        },
        "Opening date": {
            "type": "text",
            "value": ""
        },
        "Structure formula": {
            "type": "text",
            "value": ""
        }
    }
}'''

# Convert base JSON to Python dictionary
metadata_as_json = json.loads(base_json)

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    try:
        # Create an item in the correct category
        response = items_api.post_item_with_http_info(body={'category_id': category_id, 'title': row['Title']})
        # Extract the newly created item ID
        location_header_in_response = response[2].get('Location')
        print(f'The newly created item is here: {location_header_in_response}')
        item_id = int(location_header_in_response.split('/').pop())

        # Update metadata with values from the current row
        metadata_as_json = json.loads(base_json)  # Load base JSON for metadata

        # Helper function to safely assign values
        def safe_assign(field_name, row_value, default_value):
            if field_name in metadata_as_json['extra_fields']:
                metadata_as_json['extra_fields'][field_name]['value'] = row_value if pd.notnull(row_value) and row_value.strip() != '' else default_value

        safe_assign('Inventory number', row.get('Inventory number', ''), '')
        safe_assign('Substance name', row.get('Substance name', ''), '')
        safe_assign('Location', row.get('Location', ''), '')
        safe_assign('Purity', row.get('Purity', ''), 'Not specified')
        safe_assign('CAS number', row.get('CAS number', ''), '')
        safe_assign('Batch number', row.get('Batch number', ''), '')
        safe_assign('Manufacturer', row.get('Manufacturer', ''), 'Unknown')
        safe_assign('Structure formula', row.get('Structure formula', ''), 'Not specified')
        safe_assign('Opening date', row.get('Opening date', ''), 'Not specified')

        # Modify the item with the correct metadata
        items_api.patch_item(item_id, body={'title': row['Title'], 'metadata': json.dumps(metadata_as_json)})

        print(f"Item '{row['Title']}' created successfully.")
    except ApiException as e:
        print(f"An error occurred while creating item '{row['Title']}': {e}")
